# BluaDiagnostics x Care Plus — PoC Sprint 1

Este notebook demonstra:
- System prompt aplicado
- Memoria de sessao com 3+ turnos
- Function calling com pelo menos 1 tool


In [2]:
pip install anthropic python-dotenv

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp314-cp314-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/833.0 kB ? eta -:--:--
   ---------------------------------------- 833.0/833.0 kB 24.3 MB/s  0:00:00
Using cached anyio-4.13.0-py3-none-any.whl (114 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cac

In [3]:
# O símbolo de percentagem garante que a instalação vai para o Kernel ativo do seu notebook
%pip install --user tiktoken requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


In [ ]:
# Setup
import os
from ollama import Client
from dotenv import load_dotenv

# 1. Carrega as variáveis do arquivo .env de forma segura
load_dotenv()

# 2. Configuração Ollama Cloud (Exatamente como o professor pediu)
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + os.getenv('OLLAMA_API_KEY')}
)

MODEL_NAME = "gpt-oss:120b"

# 3. Função otimizada para a Sprint 1 (Memória + Tools)
def llm(mensagens, ferramentas=None, max_tokens=300, temperature=0.3):
    try:
        # A chamada principal permanece a mesma
        resposta = client.chat(
            model=MODEL_NAME,
            messages=mensagens,       # Agora aceita o histórico completo da conversa
            tools=ferramentas,        # Aceita o seu JSON de tools
            options={"num_predict": max_tokens, "temperature": temperature},
            stream=False
        )
        
        # Retorna o objeto de mensagem inteiro (crucial para o agente usar as tools)
        return resposta['message']
        
    except Exception as e:
        # Mantemos o tratamento de erro original
        return {"role": "assistant", "content": f"⚠️ Erro na conexão: {e}"}

In [ ]:
import json
import os

# 1. Carregando os Arquivos Reais (System Prompt e Tools)
caminho_prompt = os.path.join("..", "prompts", "system_prompt.md")
with open(caminho_prompt, "r", encoding="utf-8") as f:
    system_prompt = f.read()

caminho_tools = os.path.join("..", "tools", "tools_spec.json")
with open(caminho_tools, "r", encoding="utf-8") as f:
    arquivo_tools = json.load(f)
    minhas_ferramentas = arquivo_tools["tools"]

# 2. Inicializando a Memória da Conversa
historico = [
    {"role": "system", "content": system_prompt}
]

print("✅ Arquivos lidos com sucesso. Iniciando simulação...\n")
print("=" * 60)

# ==========================================
# TURNO 1: O Pedido Inicial
# ==========================================
print("\n--- INÍCIO DO TURNO 1 ---")
historico.append({
    "role": "user",
    "content": "Preciso agendar uma consulta de rotina com um cardiologista."
})
print("👤 PACIENTE: Preciso agendar uma consulta de rotina com um cardiologista.")

# Chama a IA e salva a resposta na memória
resposta_t1 = llm(mensagens=historico, ferramentas=minhas_ferramentas)
historico.append(resposta_t1) 

print(f"🤖 IA: {resposta_t1['content']}")

# ==========================================
# TURNO 2: O Function Calling (Ação Invisível)
# ==========================================
print("\n--- INÍCIO DO TURNO 2 ---")
historico.append({
    "role": "user",
    "content": "Ah, claro! Meu número de beneficiário é 987654321."
})
print("👤 PACIENTE: Ah, claro! Meu número de beneficiário é 987654321.")

# Chama a IA e salva a decisão dela na memória
resposta_t2 = llm(mensagens=historico, ferramentas=minhas_ferramentas)
historico.append(resposta_t2) 

# Verificando se a IA chamou a ferramenta (Function Calling)
if resposta_t2.get('tool_calls'):
    tool_chamada = resposta_t2['tool_calls'][0]
    nome_funcao = tool_chamada['function']['name']
    parametros = tool_chamada['function']['arguments']
    
    print("🤖 IA: [SILÊNCIO - A IA PROCESSOU A INFORMAÇÃO E DECIDIU EXECUTAR UMA FERRAMENTA]")
    print(f"⚙️ [FUNCTION CALLING EXECUTADO]: Ferramenta '{nome_funcao}' acionada!")
    print(f"⚙️ [PARÂMETROS EXTRAÍDOS]: {parametros}")
    
    # ==========================================
    # TURNO 3: O Retorno do Sistema
    # ==========================================
    print("\n--- INÍCIO DO TURNO 3 ---")
    
    # Simulando o backend do aplicativo retornando sucesso
    print(f"🔄 [SISTEMA MOCK]: Injetando resultado de sucesso da ferramenta '{nome_funcao}' na memória...")
    mensagem_sistema_mock = {
        "role": "tool",
        "name": nome_funcao,
        "content": "Sucesso. Consulta agendada para o dia 28/05 as 14h com o Dr. Roberto (Cardiologia) na clinica central."
    }
    historico.append(mensagem_sistema_mock)
    
    # Aciona a IA pela última vez
    resposta_t3 = llm(mensagens=historico, ferramentas=minhas_ferramentas)
    historico.append(resposta_t3)
    
    print(f"🤖 IA (Resposta Final Humanizada): {resposta_t3['content']}")
else:
    print(f"🤖 IA: {resposta_t2['content']}")

print("\n" + "=" * 60)
print("Prova de Conceito (PoC) finalizada com sucesso! Todos os requisitos (3 turnos + RAG/Tools) atendidos.")

✅ Arquivos lidos com sucesso. Iniciando simulação...


--- INÍCIO DO TURNO 1 ---
👤 PACIENTE: Preciso agendar uma consulta de rotina com um cardiologista.
🤖 IA: Claro! Vou agendar a sua teleconsulta com um cardiologista.

Para isso, preciso do seu **ID de beneficiário** (o número que você usa no app Care Plus). Você poderia me informar, por favor?

--- INÍCIO DO TURNO 2 ---
👤 PACIENTE: Ah, claro! Meu número de beneficiário é 987654321.
🤖 IA: [SILÊNCIO - A IA PROCESSOU A INFORMAÇÃO E DECIDIU EXECUTAR UMA FERRAMENTA]
⚙️ [FUNCTION CALLING EXECUTADO]: Ferramenta 'agendar_teleconsulta' acionada!
⚙️ [PARÂMETROS EXTRAÍDOS]: {'especialidade': 'Cardiologia', 'paciente_id': '987654321', 'urgencia': 'rotina'}

--- INÍCIO DO TURNO 3 ---
🔄 [SISTEMA MOCK]: Injetando resultado de sucesso da ferramenta 'agendar_teleconsulta' na memória...
🤖 IA (Resposta Final Humanizada): Sua teleconsulta com o cardiologista foi agendada com sucesso! 📅

- **Data:** 28/05  
- **Horário:** 14h  
- **Médico:** Dr. Roberto